# Phase 2: Evaluation Framework with Ragas

**Enterprise Agentic RAG System**

In this notebook we implement and run a production-grade evaluation pipeline using **Ragas**.

Goals:
- Load the curated evaluation dataset (`data/evaluation/eval_dataset.json`)
- Run the updated `QueryEngine` (Gemma 4 Latest + Small-to-Big retrieval on Hierarchical chunks + educational prompt) on each question
- Evaluate using Ragas metrics: faithfulness, answer_relevancy, context_precision, context_recall
- Use **Gemma 4 Latest** as the judge LLM and **nomic-embed-text** for embeddings (via LlamaIndex wrappers)
- Log timing and results
- Persist timestamped results to `artifacts/evaluation_results/`

This framework is fully configurable via `config.yaml` and repeatable.

**Note:** The QueryEngine now uses a strong educational system prompt and Small-to-Big retrieval. Re-run the cells to see new, more detailed & structured answers.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import json
import pandas as pd
from datetime import datetime

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)

Project root: c:\Users\jains\OneDrive\Desktop\RAG-SYSTEM


## 1. Load Configuration and Configure LlamaIndex

**Important for Ragas + styled tables:** If you hit `ModuleNotFoundError` for `langchain_community...` or `ImportError: background_gradient requires matplotlib`, run:

```powershell
cd "C:\Users\jains\OneDrive\Desktop\RAG-SYSTEM"
.\.venv\Scripts\python.exe -m pip install -e ".[dev]"
```

Then **restart the kernel**. This pulls in ragas + langchain providers + matplotlib for nice score tables.

In [2]:
from src.config import get_settings, settings
from src.logging_config import logger

print("=== Evaluation Configuration ===")
print("Judge LLM (Ragas)   :", settings.evaluation.llm_for_judge)
print("Embeddings (Ragas)  :", settings.evaluation.embed_model_for_ragas)
print("Metrics             :", settings.evaluation.ragas_metrics)
print("Dataset path        :", settings.evaluation.dataset_path)
print("Output directory    :", settings.evaluation.output_dir)
print("Save results        :", settings.evaluation.save_results)

# Configure global LlamaIndex Settings (Gemma 4 8B + nomic-embed-text)
settings.configure_llama_index()

from llama_index.core import Settings as LlamaSettings
print("\nLlamaIndex LLM      :", LlamaSettings.llm.model)
print("LlamaIndex Embedder :", LlamaSettings.embed_model.model_name)

2026-06-06 16:36:54.134 | INFO     | src.logging_config:setup_logging:66 - Logging initialized
c:\Users\jains\OneDrive\Desktop\RAG-SYSTEM\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== Evaluation Configuration ===
Judge LLM (Ragas)   : gemma4:latest
Embeddings (Ragas)  : nomic-embed-text
Metrics             : ['faithfulness', 'answer_relevancy']
Dataset path        : data/evaluation/eval_dataset.json
Output directory    : artifacts/evaluation_results
Save results        : True

LlamaIndex LLM      : gemma4:latest
LlamaIndex Embedder : nomic-embed-text


## 2. Load QueryEngine (from Phase 1)

In [3]:
from src.config import settings
from src.retrieval import get_query_engine
from src.evaluation import RAGASEvaluator

print("Loading QueryEngine (this connects to existing Chroma vector store)...")
# IMPORTANT: This now uses the updated implementation with:
# - Strong educational system prompt (detailed + structured Markdown answers)
# - Small-to-Big retrieval using REAL parent nodes from the docstore
#   (precise leaf matches → coherent parent section context for the LLM)
query_engine = get_query_engine()
print("QueryEngine ready using model:", settings.ollama.llm_model)
print("Using educational prompt + Small-to-Big retrieval on hierarchical chunks")

# Wire up the evaluator with the freshly created query engine
evaluator = RAGASEvaluator(query_engine=query_engine)

# Run retrieval evaluation (uses the rich metadata from Hierarchical chunking + the active retriever)
print("\n--- Retrieval Evaluation ---")
retrieval_scores = evaluator.evaluate_retrieval()
print(retrieval_scores)

2026-06-06 16:37:00.280 | INFO     | src.retrieval.query_engine:get_query_engine:100 - Creating QueryEngine


Loading QueryEngine (this connects to existing Chroma vector store)...


2026-06-06 16:37:00.754 | INFO     | src.retrieval.query_engine:get_query_engine:135 - Repopulated docstore with 3544 hierarchy nodes from sidecar for Small-to-Big / Hybrid
2026-06-06 16:37:00.755 | INFO     | src.retrieval.retriever:get_retriever:208 - Using Hybrid (Vector + BM25) with fusion_mode=reciprocal_rerank
2026-06-06 16:37:01.068 | INFO     | src.retrieval.query_engine:get_query_engine:172 - MetadataBoosterPostprocessor enabled for section bias correction
2026-06-06 16:37:01.120 | INFO     | src.evaluation.evaluator:__init__:136 - RAGAS Evaluator initialized
2026-06-06 16:37:01.121 | INFO     | src.evaluation.evaluator:evaluate_retrieval:241 - Starting retrieval evaluation over 18 questions (k=10)
2026-06-06 16:37:01.122 | INFO     | src.evaluation.evaluator:evaluate_retrieval:251 -   [1/18] Retrieving for: What is the ReAct pattern and how does it combine reasoning ...


QueryEngine ready using model: gemma4:latest
Using educational prompt + Small-to-Big retrieval on hierarchical chunks

--- Retrieval Evaluation ---


2026-06-06 16:37:02.165 | INFO     | src.evaluation.evaluator:evaluate_retrieval:296 -     pages_retrieved=[3, 27] expected=[45, 46, 47, 48] hit=False precision=1.0
2026-06-06 16:37:02.166 | INFO     | src.evaluation.evaluator:evaluate_retrieval:251 -   [2/18] Retrieving for: Explain the five levels of agentic systems as described in m...
2026-06-06 16:37:02.208 | INFO     | src.evaluation.evaluator:evaluate_retrieval:296 -     pages_retrieved=[3, 22, 31] expected=[22, 23, 24, 25] hit=True precision=0.6
2026-06-06 16:37:02.209 | INFO     | src.evaluation.evaluator:evaluate_retrieval:251 -   [3/18] Retrieving for: How does tool use and function calling enhance the capabilit...
2026-06-06 16:37:02.248 | INFO     | src.evaluation.evaluator:evaluate_retrieval:296 -     pages_retrieved=[10, 11, 29, 32, 33, 77] expected=[67, 68, 69] hit=False precision=0.6
2026-06-06 16:37:02.249 | INFO     | src.evaluation.evaluator:evaluate_retrieval:251 -   [4/18] Retrieving for: What is the difference be


=== Per-question retrieval diagnostics (smoke set) ===
Q1: hit=False pages=[3, 27] vs expected=[45, 46, 47, 48] prec=1.0
   -> page 3 | Unknown... | 25 #3) ReAct (Reason and Act) pattern...........................................
   -> page 3 | Unknown... | ...........25 #3) ReAct (Reason and Act) pattern................................
Q2: hit=True pages=[3, 22, 31] vs expected=[22, 23, 24, 25] prec=0.6
   -> page 31 | 6. Memory... | 5 Levels of Agentic AI Systems Agentic AI systems don't just generate text; they
   -> page 22 | 6. Memory... | 4) Cooperation Multi-agent systems work best when agents collaborate and exchang
Q3: hit=False pages=[10, 11, 29, 32, 33, 77] vs expected=[67, 68, 69] prec=0.6
   -> page 29 | 6. Memory... | For example, an agent in CrewAI typically alternates between reasoning about a t
   -> page 77 | #6) Setup Crew... | Let’s implement this! #1) Setup LLM We'll use a locally served DeepSeek-R1 using
Q4: hit=False pages=[9, 10, 11, 29, 42] vs expected=[112, 1

## 2.5. Quick Demo: New Educational Answers (Small-to-Big + Educational Prompt)

In [4]:
# This cell demonstrates the *new* query answers after the query_engine updates.
# Re-run this (and the evaluator below) to replace old answers with fresh educational Markdown output.

test_queries = [
    
    "Explain the 5 levels of agentic systems.",
]

for i, query in enumerate(test_queries, 1):
    print(f"\n{'='*70}")
    print(f"QUERY {i}: {query}")
    print('='*70)

    response = query_engine.query(query)
    print("\nAnswer (new educational style):")
    print(str(response)[:2000])
    print("...\n")

2026-06-06 16:37:11.934 | INFO     | src.retrieval.postprocessor:_postprocess_nodes:161 - MetadataBoosterPostprocessor applied



QUERY 1: Explain the 5 levels of agentic systems.

Answer (new educational style):
## Understanding Agentic AI Systems

As an expert educator, I can guide you through the concept of **agentic AI systems**. To understand these systems, it's helpful to think about the difference between a simple chatbot and a sophisticated, autonomous worker.

### 💡 Core Concept: What is Agency?

At its heart, **agency** refers to the ability of an AI system to act independently and achieve goals, rather than just generating text in response to a prompt.

*   **Simple Text Generation:** A basic LLM simply takes an input and generates the most statistically probable output (text).
*   **Agentic System:** An agentic system is much more capable. It can perform actions, make decisions, and execute complex, multi-step workflows autonomously (Page 31).

> **Key Takeaway:** Agentic AI systems don't just generate text; they can make decisions, call functions, and even run autonomous workflows (Page 31).

### 📚 

## 3. Preview the Evaluation Dataset

In [ ]:
dataset_path = Path(settings.paths.resolve()["evaluation"]) / "eval_dataset.json"

with open(dataset_path, "r", encoding="utf-8") as f:
    eval_dataset = json.load(f)

print(f"Total questions in dataset: {len(eval_dataset)}\n")

# Show a few examples
for i, item in enumerate(eval_dataset[:3], 1):
    print(f"--- Question {i} ---")
    print("Q:", item["question"])
    print("Ground Truth (first 200 chars):", item["ground_truth"][:200], "...")
    print()

## 4. Initialize and Run Ragas Evaluator

In [ ]:
from src.evaluation import RAGASEvaluator

evaluator = RAGASEvaluator(query_engine=query_engine)

print("Starting Ragas evaluation... (this may take several minutes with local models)")
print("Note: This will use the *new* educational query_engine + Small-to-Big retrieval.")
print("Re-running will generate fresh answers (different from any previously saved results).")
start = datetime.now()

result = evaluator.evaluate(save=True)

print(f"\nEvaluation completed in {result.duration_seconds:.1f} seconds")
print("New results saved with a fresh timestamp in artifacts/evaluation_results/")

## 5. Display Results

In [ ]:
print("=== Ragas Evaluation Summary ===")
print(f"Timestamp     : {result.timestamp}")
print(f"Model         : {result.model_used}")
print(f"Judge Model   : {result.judge_model}")
print(f"# Questions   : {result.num_questions}")
print(f"Duration      : {result.duration_seconds} seconds")
print()

print("--- Metric Scores (mean) ---")
for metric, score in sorted(result.metrics.items()):
    print(f"{metric:25s}: {score:.4f}")

In [ ]:
# Nice table using pandas
scores_df = pd.DataFrame.from_dict(result.metrics, orient="index", columns=["Score"])
scores_df.index.name = "Metric"
scores_df = scores_df.sort_values("Score", ascending=False)

print("\nRagas Scores (sorted):")
try:
    display(scores_df.style.format("{:.4f}").background_gradient(cmap="RdYlGn", vmin=0, vmax=1))
except ImportError:
    print("matplotlib not installed - showing plain table")
    display(scores_df.style.format("{:.4f}"))

## 6. Inspect Saved Artifacts

In [ ]:
output_dir = Path(settings.evaluation.output_dir)
print(f"Results saved in: {output_dir.absolute()}\n")

files = sorted(output_dir.glob(f"evaluation_{result.timestamp}*"))
for f in files:
    print(f"- {f.name}")

## 7. Load and Explore Detailed Results (if CSV was saved)

In [ ]:
import glob

detail_files = glob.glob(str(output_dir / f"evaluation_{result.timestamp}*_details.csv"))
if detail_files:
    details_df = pd.read_csv(detail_files[0])
    print("Detailed per-question scores (first 5 rows):")
    try:
        display(details_df.head().style.format(precision=3))
    except ImportError:
        print("matplotlib not installed - showing plain table")
        display(details_df.head())
else:
    print("No detailed CSV found (may happen with certain Ragas versions).")

In [5]:
import json
from pathlib import Path
from datetime import datetime

# Automatically find the latest evaluation file
eval_dir = Path("artifacts/evaluation_results")
latest_file = max(eval_dir.glob("evaluation_*.json"), key=lambda f: f.stat().st_mtime)

print(f"📄 Latest Evaluation File: {latest_file.name}\n")

with open(latest_file, "r", encoding="utf-8") as f:
    results = json.load(f)

# Print Retrieval Scores
print("=== RETRIEVAL SCORES ===")
retrieval = results.get("retrieval", {})
for key, value in retrieval.items():
    print(f"{key:25}: {value}")

print("\n=== GENERATION SCORES ===")
generation = results.get("generation", {})
if "generation_skipped" in generation:
    print(generation["generation_skipped"])
else:
    for key, value in generation.items():
        print(f"{key:25}: {value}")

📄 Latest Evaluation File: evaluation_20260606_161449.json

=== RETRIEVAL SCORES ===
recall_at_k              : 0.0694
context_precision        : 0.3333
page_hit_rate            : 0.1111
evaluated                : 18
details                  : [{'idx': 1, 'question': 'What is the ReAct pattern and how does it combine reasoning with action in AI ag', 'expected_pages': [45, 46, 47, 48], 'retrieved_pages': [3], 'hit': False, 'precision_contrib': 1.0, 'retrieved': [{'page': 3, 'section': 'Unknown', 'content_type': 'general', 'preview': '...........25 #3) ReAct (Reason and Act) pattern...........................................................26 #4) Planni'}, {'page': 3, 'section': 'Unknown', 'content_type': 'general', 'preview': '............25 #3) ReAct (Reason and Act) pattern...........................................................26 #4) Plann'}]}, {'idx': 2, 'question': 'Explain the five levels of agentic systems as described in modern agent literatu', 'expected_pages': [22, 23, 24, 2

In [ ]:
from src.ingestion import run_ingestion

num_nodes = run_ingestion(force=True)
print(f"✅ Re-ingestion completed. Total leaf nodes: {num_nodes}")

## 8. Summary & Next Steps

- ✅ Config-driven Ragas evaluation implemented
- ✅ Uses **Gemma 4 8B** as judge and **nomic-embed-text** for embeddings
- ✅ Full trajectory evaluation (answer + retrieved contexts)
- ✅ Results are logged and saved with timestamps
- ✅ Modular `RAGASEvaluator` class + convenience `run_evaluation()` function

**Typical next actions:**
- Analyze low-scoring questions and improve retrieval/chunking/prompts
- Add more metrics or custom metrics (e.g. citation accuracy)
- Run evaluation regularly as a regression test after pipeline changes
- Compare different retrieval strategies (hybrid, reranking, etc.) using the same dataset
- Integrate evaluation into CI or a simple CLI command (`rag evaluate`)

The evaluation framework is now ready for systematic improvement of the agentic RAG system.